# Load the Annotations
Drop the unused columns

In [16]:
import pandas as pd 
data = pd.read_csv("./worm_cat/static/download/annotation061319b.csv") 
#data.drop(data.columns[[6,7,8,9,10]], axis=1, inplace=True) 
#del data['Unnamed: 6']
data['index_col'] = data.index
data.head()

,Sequence ID,Wormbase ID,Category 1,Category 2,Category 3,Automated Description,index_col
0,F15H10.3,WBGene00000144,Cell cycle,Cell cycle: APC,Cell cycle: APC,apc-10 is an ortholog of human ANAPC10 (anapha...,0
1,F35G12.9,WBGene00000145,Cell cycle,Cell cycle: APC,Cell cycle: APC,apc-11 is an ortholog of human ANAPC11 (anapha...,1
2,C09H10.7,WBGene00007501,Cell cycle,Cell cycle: APC,Cell cycle: APC,apc-17 is an ortholog of human ANAPC1 (anaphas...,2
3,K06H7.6,WBGene00000143,Cell cycle,Cell cycle: APC,Cell cycle: APC,apc-2 is an ortholog of human ANAPC2 (anaphase...,3
4,B0511.9,WBGene00015235,Cell cycle,Cell cycle: APC,Cell cycle: APC,cdc-26 is expressed in the germ line.,4


# Check for Empty Categories

In [18]:
print("=========================================")
print("=========      Category 1      ==========")
print(data[data['Category 1'].isnull()]['Wormbase ID'])
print("=========================================")
print("=========      Category 2      ==========")
print(data[data['Category 2'].isnull()]['Wormbase ID'])
print("=========================================")
print("=========      Category 3      ==========")
print(data[data['Category 3'].isnull()]['Wormbase ID'])



=========      Category 1      ==========
Series([], Name: Wormbase ID, dtype: object)
=========      Category 2      ==========
Series([], Name: Wormbase ID, dtype: object)
=========      Category 3      ==========
Series([], Name: Wormbase ID, dtype: object)


# Check for Duplicate Wormbase ID's

In [19]:
ids = data['Wormbase ID']
dup_ids = ids[ids.duplicated()]
#print(dup_ids)
df_dup_ids = pd.DataFrame({'Index':dup_ids.index, 'Wormbase ID':dup_ids.values})
print("Full List={} Dups={}".format(len(ids.index),len(dup_ids.index)))
df_dup_ids = df_dup_ids.sort_values('Wormbase ID')
total_dups = pd.DataFrame()
for i, row in df_dup_ids.iterrows():
    dup_row = data.loc[data['Wormbase ID'] == row['Wormbase ID']]
    dup_row = dup_row[['Sequence ID','Wormbase ID','Category 1']]
    total_dups = total_dups.append(dup_row)

pd.set_option('display.max_rows', 400)    
print(total_dups)
print('==================')
    

Full List=31490 Dups=0
Empty DataFrame
Columns: []
Index: []


# Find Differences in Category 1 and Catergory 2
Each Category 1 Item should start with the Same text in Category 2
Check each category and sum the count if they match we are good
if they do not further investivation is needed.

In [20]:
cat1 = data['Category 1'].unique()

series = data.groupby('Category 1')['index_col'].nunique()
unique_cat1 = pd.DataFrame({'Category 1':series.index, 'Count':series.values})
#print("type {}".format(unique_cat1))

print(" Cat1  Cat2  Diff Category 1")
for i, row in unique_cat1.iterrows():
    regex = "^{}".format(row['Category 1'])
    count = data.set_index('Category 2').filter(regex=regex, axis=0)['Wormbase ID'].count()
    if(count != row['Count']):
        print("{0: >5} {1: >5}  {2: >4} '{3}'".format(row['Count'],count,(count-row['Count']),row['Category 1']))

        

 Cat1  Cat2  Diff Category 1


# Let's look at Extracellular material

In [21]:
unique_2 = data.set_index('Category 2').filter(regex='^Extracellular material', axis=0)
#unique_2.set_index('index_col', inplace=True)
unique_1 = data.set_index('Category 1').filter(regex='^Extracellular material', axis=0)
print("Unique Cat 1={0:} Cat 2={1:}".format(len(unique_1.index),len(unique_2.index)))
count=0
for i, row in unique_1.iterrows():
    dup_row = unique_2.loc[unique_2['index_col'] == row['index_col']]
    if(dup_row.empty):
        count +=1
        print("{0:>2} {1:}".format(count,row[1]))
    

        

Unique Cat 1=483 Cat 2=483


# Let's look at Transcription: other

In [6]:
unique_2 = data.set_index('Category 2').filter(regex='^Transcription: other', axis=0)
#unique_2.set_index('index_col', inplace=True)
unique_1 = data.set_index('Category 1').filter(regex='^Transcription: other', axis=0)
print("{0: >5} {1: >5}".format(len(unique_1.index),len(unique_2.index)))
for i, row in unique_2.iterrows():
    dup_row = unique_1.loc[unique_1['index_col'] == row['index_col']]
    if(dup_row.empty):
        print(row)
   

   14    14


# Find Differences in Category 2 and Catergory 3



In [22]:
cat2 = data['Category 2'].unique()
#print(cat2)
series = data.groupby('Category 2')['Sequence ID'].nunique()
df = pd.DataFrame({'Category 2':series.index, 'Count':series.values})

print(" Cat2  Cat3  Diff Category 2")
for i, row in df.iterrows():
    regex = "^{}".format(row['Category 2'])
    count = data.set_index('Category 3').filter(regex=regex, axis=0)['Sequence ID'].count()
    if(count-row['Count']<0):
        print("{0: >5} {1: >5}  {2: >4} '{3}'".format(row['Count'],count,(count-row['Count']),row['Category 2']))
    if(count-row['Count']>0):
        print("{0: >5} {1: >5}  {2: >4} '{3}'".format(row['Count'],count,(count-row['Count']),row['Category 2']))

        

 Cat2  Cat3  Diff Category 2
    1     2     1 'Proteolysis proteosome: E1'
    8    46    38 'Proteolysis proteosome: ubiquitin'
    5   138   133 'Transcription: general machinery: RNA Pol I'
  122   133    11 'Transcription: general machinery: RNA Pol II'
   10    19     9 'Transmembrane transport: ion'
 8190  8201    11 'Unknown'


In [27]:
unique_3 = data.set_index('Category 3').filter(regex='^Unknown', axis=0)
unique_2 = data.set_index('Category 2').filter(regex='^Unknown$', axis=0)
print("{0: >5} {1: >5}".format(len(unique_2.index),len(unique_3.index)))
for i, row in unique_2.iterrows():
        row_data = data.loc[data['index_col'] == row['index_col']]
        #print(row_data['Category 3'])
   

 8190  8201


# List of unique categories in 1, 2, 3

In [23]:
cat1 = data['Category 1'].unique()
for c in cat1:
    print(c)

Cell cycle
Chaperone
Cilia
Cytoskeleton
Development
DNA
Extracellular material
Globin
Lysosome
Major sperm protein
Metabolism
mRNA functions
Muscle function
Neuronal function
Non-coding RNA
Nuclear pore
Nucleic acid
Peroxisome
Protein modification
Proteolysis general
Proteolysis proteosome
Pseudogene
Ribosome
Signaling
Stress response
Trafficking
Transcription factor
Transcription: chromatin
Transcription: dosage compensation
Transcription: general machinery
Transcription: other
Transmembrane protein
Transmembrane transport
Unknown


In [11]:
pd.set_option('display.max_rows', 400) 
cat2 = data['Category 2'].unique()
for c in cat2:
    print(c)

Cell cycle: APC
Cell cycle: chromosome dynamics
Cell cycle: cyclin
Cell cycle: kinase
Cell cycle: meiotic/mitotic spindle
Cell cycle: other
Cell cycle: phosphatase
Cell cycle: transcriptional regulator
Chaperone: cyclophillin
Chaperone: DnaJ domain
Chaperone: HSP
Chaperone: other
Chaperone: T complex
Cilia: IFT
Cilia: other
Cilia: transtion zone
Cytoskeleton: actin function
Cytoskeleton: cadherin
Cytoskeleton: centrosome
Cytoskeleton: claudin
Cytoskeleton: innexin
Cytoskeleton: integrin
Cytoskeleton: intermediate filament protein
Cytoskeleton: microtubule
Cytoskeleton: motor protein
Cytoskeleton: other
Cytoskeleton: paxillin
Development: apoptosis
Development: general
Development: germline
Development: sex determination
Development: somatic
DNA: helicase
DNA: nuclease
DNA: repair
DNA: replication
DNA: telomere
DNA: transposon function
Extracellular material: chitinase
Extracellular Material: chondriotin sulfostansferase
Extracellular material: collagen
Extracellular material: cuticlin


In [12]:
cat3 = data['Category 3'].unique()
for c in cat3:
    print(c)

Cell cycle: APC
Cell cycle: chromosome dynamics: CENP
Cell cycle: chromosome dynamics: condensin
Cell cycle: chromosome dynamics: kinase
Cell cycle: chromosome dynamics: kinetochore protein
Cell cycle: chromosome dynamics: meiotic functions
Cell cycle: chromosome dynamics: other
Cell cycle: chromosome dynamics: SMC
Cell cycle: cyclin
Cell cycle: kinase
Cell cycle: meiotic/mitotic spindle
Cell cycle: other
Cell cycle: phosphatase
Cell cycle: transcriptional regulator
Chaperone: cyclophillin
Chaperone: DnaJ domain
Chaperone: HSP
Chaperone: other
Chaperone: T complex
Cilia: IFT: BBSome
Cilia: IFT-A
Cilia: IFT-B
Cilia: IFT-B 
Cilia: IFT-motor
Cilia: other
Cilia: transtion zone: MKS module
Cilia: transtion zone: NPHP module
Cilia: transtion zone: other
Cytoskeleton: actin function: actin
Cytoskeleton: actin function: binding
Cytoskeleton: cadherin
Cytoskeleton: centrosome: centriole
Cytoskeleton: centrosome: kinase
Cytoskeleton: centrosome: other
Cytoskeleton: claudin
Cytoskeleton: innexin
